<a href="https://colab.research.google.com/github/syauqidamario/thesis-MTI-syauqi/blob/main/01_data_berita.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pygooglenews pandas tqdm
from pygooglenews import GoogleNews
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm # Untuk melihat progress bar
import time

gn = GoogleNews(lang='id', country='ID')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.9 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=e9452feb83be4e2afe2efd9310ecc791171c5d3efe1f3bc5280b69d263ea988a
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [2]:
# Membuat daftar rentang tanggal bulanan dari 2020 sampai 2025
start_date = datetime(2020, 1, 1)
end_date = datetime(2025, 12, 31)

def generate_monthly_intervals(start, end):
    intervals = []
    current = start
    while current < end:
        next_month = (current.replace(day=28) + timedelta(days=4)).replace(day=1)
        intervals.append((current.strftime('%Y-%m-%d'), next_month.strftime('%Y-%m-%d')))
        current = next_month
    return intervals

intervals = generate_monthly_intervals(start_date, end_date)

In [3]:
# Scraping berita
all_news = []

for start, end in tqdm(intervals, desc="Scraping Berita"):
    query = f'"BBCA" saham after:{start} before:{end}'
    search = gn.search(query)

    for entry in search['entries']:
        all_news.append({
            'raw_date': entry.published,
            'title': entry.title,
            'source': entry.source.get('title'),
            'link': entry.link
        })
    # Beri jeda sedikit agar tidak dianggap bot oleh Google
    time.sleep(1)

df_raw = pd.DataFrame(all_news)

Scraping Berita: 100%|██████████| 72/72 [01:32<00:00,  1.28s/it]


In [4]:
# menampilkan data
df_raw.head(5)

,raw_date,title,source,link
0,"Wed, 15 Jan 2020 08:00:00 GMT",Harga Saham BCA Cetak Rekor Baru - CNBC Indonesia,CNBC Indonesia,https://news.google.com/rss/articles/CBMimgFBV...
1,"Sat, 18 Jan 2020 08:00:00 GMT","Top, Saham BBRI Catat Rekor Tertinggi di Tahun...",Ajaib,https://news.google.com/rss/articles/CBMijAFBV...
2,"Wed, 01 Jan 2020 08:00:00 GMT",Saham-saham yang Paling Untung dan Buntung Sep...,Katadata.co.id,https://news.google.com/rss/articles/CBMirwFBV...
3,"Thu, 20 Feb 2020 08:00:00 GMT","Laba BCA 2019 Tembus Rp 28,6 T, Melesat 11% - ...",CNBC Indonesia,https://news.google.com/rss/articles/CBMipAFBV...
4,"Fri, 21 Feb 2020 08:00:00 GMT",Bursa Sore: IHSG Dan Rupiah Sama-sama Tergelin...,Indo Premier Sekuritas,https://news.google.com/rss/articles/CBMiyAJBV...


In [5]:
def clean_news_title(text):
    # Menghapus nama sumber berita
    if ' - ' in text:
        text = text.rsplit(' - ', 1)[0]
    # Lowercase
    text = text.lower()
    # Menghapus karakter non-alfanumeric
    import re
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.strip()

df_raw['cleaned_title'] = df_raw['title'].apply(clean_news_title)
df_raw['cleaned_title'].head(5)

,cleaned_title
0,harga saham bca cetak rekor baru
1,top saham bbri catat rekor tertinggi di tahun ...
2,sahamsaham yang paling untung dan buntung sepa...
3,laba bca 2019 tembus rp 286 t melesat 11
4,bursa sore ihsg dan rupiah samasama tergelincir


In [6]:
# menyelaraskan waktu dan akumulasi akhir pekan

# Konversi ke format datetime
df_raw['Date'] = pd.to_datetime(df_raw['raw_date']).dt.tz_localize(None)

def adjust_to_trading_day(date):
    # 5 = Sabtu, 6 = Minggu
    if date.weekday() == 5: # Sabtu
        return (date + timedelta(days=2)).date()
    elif date.weekday() == 6: # Minggu
        return (date + timedelta(days=1)).date()
    else:
        return date.date()

df_raw['trading_date'] = df_raw['Date'].apply(adjust_to_trading_day)
df_raw['trading_date'].head(5)

,trading_date
0,2020-01-15
1,2020-01-20
2,2020-01-01
3,2020-02-20
4,2020-02-21


In [7]:
# agregasi harian (finalisasi dataset)
# Menggabungkan berita per tanggal perdagangan
df_final = df_raw.groupby('trading_date').agg({
    'cleaned_title': lambda x: ' [SEP] '.join(x), # Gunakan pemisah [SEP]
    'link': 'count'
}).reset_index()

df_final.columns = ['date', 'aggregated_news', 'news_count']

# Simpan hasil untuk Notebook 02
df_final.to_csv('01_berita_bbca_prepared.csv', index=False)